# SkinHybrid: A High-Accuracy Hybrid Model for 8-Class Dermoscopic Classification

**Goal.** Maximise accuracy honestly, ablate every design choice, and end with one
well-justified hybrid model.

### Honest targets

| Setting | Top-1 | Balanced acc. | Note |
|---|---|---|---|
| Your original MorphoFire | 0.620 | 0.575 | ResNet50+MNv3, 224px, stacked imbalance fixes |
| This notebook, baseline | ~0.78 | ~0.65 | ConvNeXt-Tiny, 224px, modern recipe |
| This notebook, final | **~0.83** | **~0.72** | 384px + EMA + TTA + 3-seed ensemble |
| ISIC 2019 challenge winner | — | 0.636 | on the *official* test set, large ensemble |

The winner's 0.636 is on the official held-out test set, which is harder than any
internal split. Numbers above 0.95 in the literature almost always come from
image-level splits that scatter duplicate photographs of one lesion across train and
test. We split at **lesion level**, so our numbers are lower and real.

### The seven levers that actually raise accuracy
1. **Resolution.** 384px instead of 224px. Dermoscopic criteria are fine-grained.
   Usually the single largest gain.
2. **Backbone.** ConvNeXt-Tiny / EfficientNetV2-S instead of ResNet50.
3. **One principled imbalance fix.** Balanced-softmax, not sampler + focal + alpha +
   smoothing stacked together (that wrecks the operating point, which is why your
   precision collapsed on rare classes).
4. **Modern augmentation.** RandAugment + mixup/CutMix.
5. **EMA** of weights.
6. **TTA.** 8x dihedral at inference, essentially free accuracy.
7. **Ensembling** across seeds and backbones.

### The hybrid, done properly
Your original concatenated 5 morphological dims onto 3328 deep dims (0.15% of the
vector) and never ablated it. Here the auxiliary branches are **projected and gated
(FiLM)** so the network can actually use or ignore them, and we test two branches:

* **Morphology** (16 handcrafted shape/border/colour descriptors)
* **Patient metadata** (age, sex, anatomical site) — the winning ISIC 2019 entry used
  metadata, so this is the branch most likely to earn its place

Section 14 answers which one helps, with a placebo control.

---
**Hardware:** RTX 3060 Ti (8 GB). Ablation at 224px (~5 h), final models at 384px
with gradient accumulation (~10 h). Everything is checkpointed and resumable.


## 1. Configuration


In [1]:
from pathlib import Path
import os, sys, json, time, math, random, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

class CFG:
    DATA_ROOT = Path("/content/data" if IN_COLAB else "./data")
    OUT_ROOT  = Path("/content/outputs" if IN_COLAB else "./outputs")

    IMG_DIR   = DATA_ROOT / "ISIC_2019_Training_Input"
    GT_CSV    = DATA_ROOT / "ISIC_2019_Training_GroundTruth.csv"
    META_CSV  = DATA_ROOT / "ISIC_2019_Training_Metadata.csv"
    HAM_META  = DATA_ROOT / "HAM10000_metadata.csv"
    CACHE_DIR = DATA_ROOT / "cache_512"     # 512 short side so 384 training is possible

    # ---- ablation phase (fast) ----
    ABL_SIZE, ABL_EPOCHS, ABL_BS = 224, 18, 32
    # ---- final phase (accurate) ----
    FIN_SIZE, FIN_EPOCHS, FIN_BS, FIN_ACCUM = 384, 24, 12, 3   # effective batch 36

    LR, WD, WARMUP = 3e-4, 0.05, 2
    EMA_DECAY = 0.999
    MIXUP_ALPHA, CUTMIX_ALPHA, MIX_PROB = 0.2, 1.0, 0.5
    LABEL_SMOOTH = 0.05
    NUM_WORKERS = 0 if os.name == "nt" else 4
    SEEDS = [0, 1, 2]

    CLASSES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
    NUM_CLASSES = 8

for s in ["figures","tables","checkpoints","preds"]:
    (CFG.OUT_ROOT / s).mkdir(parents=True, exist_ok=True)
print("data:", CFG.DATA_ROOT.resolve(), "\nout :", CFG.OUT_ROOT.resolve())


data: E:\Fabu\dermaudit\SkinHybrid\data 
out : E:\Fabu\dermaudit\SkinHybrid\outputs


In [2]:
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (balanced_accuracy_score, f1_score, roc_auc_score,
                             precision_recall_fscore_support, confusion_matrix)

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False
    print("timm not found -> `pip install timm` for ConvNeXt/EffNetV2. "
          "Falling back to torchvision backbones.")

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | {DEV}")
if DEV.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


torch 2.5.1+cu121 | cuda
NVIDIA GeForce RTX 3060 Ti 8.6 GB


## 2. Manifest: labels, lesion IDs, patient metadata

Get the data with `python download_data.py --root ./data` (official ISIC S3 +
Harvard Dataverse). `ISIC_2019_Training_Metadata.csv` gives both the patient
metadata for the hybrid branch **and** `lesion_id` for honest splitting.


In [3]:
def build_manifest():
    gt = pd.read_csv(CFG.GT_CSV)
    gt = gt.rename(columns={gt.columns[0]: "image"})
    cols = [c for c in CFG.CLASSES if c in gt.columns]
    assert len(cols) == 8, f"found class columns: {cols}"
    df = pd.DataFrame({"image": gt["image"], "label": gt[cols].values.argmax(1)})
    df["label_name"] = [CFG.CLASSES[i] for i in df.label]
    df["path"] = df.image.map(lambda x: str(CFG.IMG_DIR / f"{x}.jpg"))

    df["lesion_id"] = pd.NA
    df["age"] = np.nan; df["sex"] = "unknown"; df["site"] = "unknown"
    if CFG.META_CSV.exists():
        m = pd.read_csv(CFG.META_CSV).rename(columns=lambda c: c.strip())
        m = m.rename(columns={m.columns[0]: "image"})
        m = m.set_index("image")
        for src, dst in [("lesion_id","lesion_id"), ("age_approx","age"),
                         ("sex","sex"), ("anatom_site_general","site")]:
            if src in m.columns:
                df[dst] = df.image.map(m[src])
        print(f"metadata joined | lesion_id on {df.lesion_id.notna().sum()}/{len(df)}")
    if CFG.HAM_META.exists():
        sep = "\t" if CFG.HAM_META.suffix == ".tab" else ","
        h = pd.read_csv(CFG.HAM_META, sep=sep)
        lut = dict(zip(h.image_id, h.lesion_id))
        df["lesion_id"] = df.lesion_id.fillna(
            df.image.map(lambda i: f"HAM_{lut[i]}" if i in lut else pd.NA))
    df["sex"] = df.sex.fillna("unknown").astype(str).str.lower()
    df["site"] = df.site.fillna("unknown").astype(str).str.lower()
    print(f"lesion_id total coverage: {df.lesion_id.notna().sum()}/{len(df)}")
    print(df.label_name.value_counts().reindex(CFG.CLASSES).to_string())
    return df

man = build_manifest()


metadata joined | lesion_id on 23247/25331
lesion_id total coverage: 23247/25331
label_name
MEL      4522
NV      12875
BCC      3323
AK        867
BKL      2624
DF        239
VASC      253
SCC       628


## 3. Resize cache (512px short side)
One-off, ~20 min, ~4 GB. BCN20000 ships 1024x1024 JPEGs; decoding those every epoch
leaves the GPU idle. 512 lets us train at both 224 and 384 from the same cache.


In [4]:
def build_cache(df, size=512, q=93):
    CFG.CACHE_DIR.mkdir(parents=True, exist_ok=True)
    todo = [(p, str(CFG.CACHE_DIR/Path(p).name)) for p in df.path
            if not (CFG.CACHE_DIR/Path(p).name).exists()]
    print(f"cache: {len(df)-len(todo)} present, {len(todo)} to build")
    if todo:
        from concurrent.futures import ThreadPoolExecutor
        def one(j):
            s, d = j
            im = cv2.imread(s, cv2.IMREAD_COLOR)
            if im is None: return s
            h, w = im.shape[:2]; sc = size/min(h, w)
            if sc < 1.0:
                im = cv2.resize(im, (int(w*sc), int(h*sc)), interpolation=cv2.INTER_AREA)
            cv2.imwrite(d, im, [cv2.IMWRITE_JPEG_QUALITY, q]); return None
        t0 = time.time()
        with ThreadPoolExecutor(max_workers=8) as ex:
            bad = [r for r in ex.map(one, todo) if r]
        print(f"built in {time.time()-t0:.0f}s ({len(bad)} failures)")
    df["cpath"] = [str(CFG.CACHE_DIR/Path(p).name)
                   if (CFG.CACHE_DIR/Path(p).name).exists() else p for p in df.path]
    return df

man = build_cache(man)


cache: 25331 present, 0 to build


## 4. Lesion-level splits
Multiple photographs of one lesion must never straddle the split. We assert zero
overlap; if that assertion fires, every number afterwards would be inflated.


In [5]:
def phash64(p, hs=8, hf=4):
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if im is None: return None
    im = cv2.resize(im, (hs*hf, hs*hf), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(im)[:hs, :hs]
    b = (d > np.median(d[1:,1:])).flatten(); o = 0
    for x in b: o = (o << 1) | int(x)
    return o

class UF:
    def __init__(s,n): s.p=list(range(n))
    def find(s,x):
        while s.p[x]!=x: s.p[x]=s.p[s.p[x]]; x=s.p[x]
        return x
    def union(s,a,b):
        ra,rb=s.find(a),s.find(b)
        if ra!=rb: s.p[rb]=ra

def assign_groups(df, radius=6):
    cache = CFG.OUT_ROOT/"tables"/"phash.csv"
    if cache.exists(): h = pd.read_csv(cache)
    else:
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=8) as ex:
            hs = list(ex.map(phash64, df.cpath.tolist()))
        h = pd.DataFrame({"image": df.image.values, "phash": hs}); h.to_csv(cache, index=False)
    df = df.merge(h, on="image", how="left")
    uf = UF(len(df)); idx = {im:i for i,im in enumerate(df.image)}
    for _, g in df[df.lesion_id.notna()].groupby("lesion_id"):
        ids=[idx[i] for i in g.image]
        for j in ids[1:]: uf.union(ids[0], j)
    buck={}
    for i,ph in enumerate(df.phash.values):
        if ph is None or pd.isna(ph): continue
        buck.setdefault(int(ph)>>48, []).append(i)
    for _, mem in buck.items():
        if not (2<=len(mem)<=400): continue
        for a in range(len(mem)):
            for b in range(a+1,len(mem)):
                if bin(int(df.phash.iloc[mem[a]])^int(df.phash.iloc[mem[b]])).count("1")<=radius:
                    uf.union(mem[a],mem[b])
    df["group_id"]=pd.factorize(np.array([uf.find(i) for i in range(len(df))]))[0]
    sz=df.groupby("group_id").size()
    print(f"{len(df)} images -> {df.group_id.nunique()} lesion groups; "
          f"{(sz>1).sum()} groups hold >1 image")
    return df

man = assign_groups(man)

def make_splits(df, seed=42):
    k = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)
    f = list(k.split(np.zeros(len(df)), df.label.values, groups=df.group_id))
    te, va = f[0][1], f[1][1]
    m = np.ones(len(df), bool); m[te]=False; m[va]=False
    sp = {"train": np.where(m)[0], "val": va, "test": te}
    tr = set(df.iloc[sp["train"]].group_id)
    for s in ["val","test"]:
        n = df.iloc[sp[s]].group_id.isin(tr).sum()
        assert n == 0, f"LEAK: {n} {s} images share a lesion with train"
    print("splits:", {k2: len(v) for k2, v in sp.items()}, "| zero overlap asserted")
    return sp

SP = make_splits(man)
man.to_csv(CFG.OUT_ROOT/"tables"/"manifest.csv", index=False)


25331 images -> 13311 lesion groups; 4721 groups hold >1 image
splits: {'train': 16887, 'val': 4222, 'test': 4222} | zero overlap asserted


## 5. Auxiliary features for the hybrid branches

**Metadata** (age, sex, site) is one-hot / normalised. Missing values get an explicit
`unknown` category rather than silent imputation, because missingness is itself
informative in ISIC.

**Morphology**: the 16 handcrafted descriptors. We also flag masks that touch the
image border, since inverse Otsu on a full frame can lock onto dermoscope vignetting
instead of the lesion, which would corrupt every shape descriptor.


In [6]:
SITES = ["anterior torso","upper extremity","posterior torso","lower extremity",
         "lateral torso","head/neck","palms/soles","oral/genital","unknown"]
SEXES = ["male","female","unknown"]

def build_metadata(df):
    age = df.age.astype(float)
    a = age.fillna(age.median()).clip(0,90)/90.0
    miss = age.isna().astype(float)
    sx = pd.get_dummies(pd.Categorical(df.sex, categories=SEXES), dtype=float)
    st = pd.get_dummies(pd.Categorical(df.site.where(df.site.isin(SITES), "unknown"),
                                       categories=SITES), dtype=float)
    M = np.concatenate([a.values[:,None], miss.values[:,None], sx.values, st.values], 1)
    print(f"metadata vector: {M.shape[1]} dims "
          f"(age, age-missing, {len(SEXES)} sex, {len(SITES)} site)")
    print(f"age missing: {100*miss.mean():.1f}% | site unknown: "
          f"{100*(~df.site.isin(SITES[:-1])).mean():.1f}%")
    return M.astype(np.float32)

META = build_metadata(man)

DESCR = ["area_ratio","perim_ratio","compactness","eccentricity","erosion_ratio",
         "dilation_ratio","ero_dil_ratio","hitmiss_ratio","ellipse_fit",
         "mean_B","mean_G","mean_R","std_B","std_G","std_R","log_mean"]

def descriptors(path):
    im = cv2.imread(path, cv2.IMREAD_COLOR)
    if im is None: return None
    H,W = im.shape[:2]
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    _, mk = cv2.threshold(cv2.GaussianBlur(g,(5,5),0), 0, 255,
                          cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
    B = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    mk = cv2.morphologyEx(mk, cv2.MORPH_CLOSE, B)
    cs,_ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return None
    c = max(cs, key=cv2.contourArea)
    A,P = cv2.contourArea(c), cv2.arcLength(c,True)
    if A<=0 or P<=0: return None
    ero,dil = cv2.erode(mk,B), cv2.dilate(mk,B)
    hm = np.zeros_like(mk)
    k0 = np.array([[-1,-1,0],[-1,1,1],[0,1,0]], np.int8)
    for r in range(4): hm |= cv2.morphologyEx(mk, cv2.MORPH_HITMISS, np.rot90(k0,r))
    ne,nd = int((ero>0).sum()), int((dil>0).sum())
    ecc=fit=0.0
    if len(c)>=5:
        (_,_),(MA,ma),_ = cv2.fitEllipse(c); lo,hi = min(MA,ma), max(MA,ma)
        if hi>0:
            ecc = float(np.sqrt(max(0.,1-(lo/hi)**2)))
            Ae = np.pi*(MA/2)*(ma/2); fit = float(min(A,Ae)/max(A,Ae)) if Ae>0 else 0.
    m = mk>0
    mu=[float(im[:,:,i][m].mean()) if m.any() else 0. for i in range(3)]
    sd=[float(im[:,:,i][m].std())  if m.any() else 0. for i in range(3)]
    lg=(255/np.log(256))*np.log1p(g.astype(np.float32))
    vals=[A/(H*W), P/(H+W), 4*np.pi*A/(P*P), ecc, ne/(H*W), nd/(H*W), ne/max(nd,1),
          float((hm>0).sum())/(H*W), fit, mu[0]/255, mu[1]/255, mu[2]/255,
          sd[0]/255, sd[1]/255, sd[2]/255, (float(lg[m].mean()) if m.any() else 0.)/255]
    return dict(zip(DESCR, vals)) | {"border": bool(mk[0,:].any() or mk[-1,:].any()
                                                    or mk[:,0].any() or mk[:,-1].any())}

def build_morph(df):
    f = CFG.OUT_ROOT/"tables"/"morph.csv"
    if f.exists(): out = pd.read_csv(f)
    else:
        from concurrent.futures import ThreadPoolExecutor
        t0=time.time()
        with ThreadPoolExecutor(max_workers=8) as ex:
            rows = list(ex.map(descriptors, df.cpath.tolist()))
        ok=[(i,r) for i,r in zip(df.image,rows) if r is not None]
        out=pd.DataFrame([r for _,r in ok]); out.insert(0,"image",[i for i,_ in ok])
        out.to_csv(f,index=False); print(f"descriptors in {time.time()-t0:.0f}s")
    d = df[["image"]].merge(out, on="image", how="left")
    for c in DESCR: d[c] = d[c].fillna(d[c].median())
    print(f"masks touching border: {100*d['border'].fillna(False).mean():.1f}% "
          "(high => Otsu is grabbing vignetting, shape descriptors unreliable)")
    return d[DESCR].values.astype(np.float32)

MORPH = build_morph(man)
print("morph:", MORPH.shape, "| meta:", META.shape)


metadata vector: 14 dims (age, age-missing, 3 sex, 9 site)
age missing: 1.7% | site unknown: 10.4%
masks touching border: 82.8% (high => Otsu is grabbing vignetting, shape descriptors unreliable)
morph: (25331, 16) | meta: (25331, 14)


## 6. Dataset and augmentation

RandAugment plus dihedral flips. Dermoscopic images have no canonical orientation, so
the full 8-element dihedral group is a free, label-preserving symmetry, which is also
why 8x TTA works so well later.


In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

def make_tf(size, train):
    if train:
        return T.Compose([
            T.ToPILImage(),
            T.RandomResizedCrop(size, scale=(0.6,1.0), ratio=(0.75,1.33)),
            T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
            T.RandAugment(num_ops=2, magnitude=9),
            T.ToTensor(), T.Normalize(MEAN, STD),
            T.RandomErasing(p=0.25, scale=(0.02,0.15)),
        ])
    return T.Compose([T.ToPILImage(), T.Resize(int(size*1.15)), T.CenterCrop(size),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

class SkinDS(Dataset):
    def __init__(self, df, idx, morph, meta, size, train):
        self.d = df.iloc[idx].reset_index(drop=True)
        self.m = morph[idx]; self.t = meta[idx]
        self.y = self.d.label.values.astype(np.int64)
        self.tf = make_tf(size, train)
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        im = cv2.imread(self.d.cpath.iloc[i], cv2.IMREAD_COLOR)
        im = np.zeros((64,64,3), np.uint8) if im is None else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        return (self.tf(im), torch.from_numpy(self.m[i]),
                torch.from_numpy(self.t[i]), int(self.y[i]))

def make_loader(df, idx, size, train, bs, morph=None, meta=None):
    ds = SkinDS(df, idx, MORPH if morph is None else morph,
                META if meta is None else meta, size, train)
    return DataLoader(ds, batch_size=bs, shuffle=train, drop_last=train,
                      num_workers=CFG.NUM_WORKERS, pin_memory=True,
                      persistent_workers=CFG.NUM_WORKERS > 0)


## 7. mixup / CutMix
Strong regulariser on a 25k-image set and one of the more reliable gains here.


In [ ]:
def rand_bbox(H, W, lam):
    r = math.sqrt(1-lam); ch, cw = int(H*r), int(W*r)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = np.clip(cy-ch//2, 0, H), np.clip(cy+ch//2, 0, H)
    x1, x2 = np.clip(cx-cw//2, 0, W), np.clip(cx+cw//2, 0, W)
    return y1, y2, x1, x2

def mix_batch(x, y):
    """Returns x, (y_a, y_b, lam). lam=1 means no mixing was applied."""
    if np.random.rand() > CFG.MIX_PROB:
        return x, (y, y, 1.0)
    perm = torch.randperm(x.size(0), device=x.device)
    if np.random.rand() < 0.5:
        lam = np.random.beta(CFG.MIXUP_ALPHA, CFG.MIXUP_ALPHA)
        x = lam*x + (1-lam)*x[perm]
    else:
        lam = np.random.beta(CFG.CUTMIX_ALPHA, CFG.CUTMIX_ALPHA)
        y1,y2,x1,x2 = rand_bbox(x.size(2), x.size(3), lam)
        x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1 - ((y2-y1)*(x2-x1)/(x.size(2)*x.size(3)))
    return x, (y, y[perm], float(lam))

def mix_loss(crit, out, target):
    ya, yb, lam = target
    return lam*crit(out, ya) + (1-lam)*crit(out, yb)


## 8. Model: backbone + FiLM-gated auxiliary branches

The fix for your original design. Instead of concatenating 5 raw dims onto 3328 deep
dims, each auxiliary branch is **projected to a comparable width and used to modulate
the deep features** (FiLM: feature-wise linear modulation). The network can amplify or
switch the branch off, so if the branch is useless we learn that rather than diluting it.


In [ ]:
BACKBONES = {
    "convnext_tiny":   ("convnext_tiny.fb_in22k_ft_in1k", 768),
    "effnetv2_s":      ("tf_efficientnetv2_s.in21k_ft_in1k", 1280),
    "swin_tiny":       ("swin_tiny_patch4_window7_224.ms_in22k_ft_in1k", 768),
    "resnet50":        ("resnet50.a1_in1k", 2048),
}

def make_backbone(name, pretrained=True):
    if HAS_TIMM and name in BACKBONES:
        mid, dim = BACKBONES[name]
        m = timm.create_model(mid, pretrained=pretrained, num_classes=0)
        return m, m.num_features
    m = torchvision.models.resnet50(
        weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Identity()
    return m, 2048

class FiLM(nn.Module):
    """Auxiliary vector -> per-channel scale and shift on the deep feature."""
    def __init__(self, aux_dim, deep_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(nn.BatchNorm1d(aux_dim), nn.Linear(aux_dim, hidden),
                                 nn.GELU(), nn.Linear(hidden, 2*deep_dim))
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, deep, aux):
        g, b = self.net(aux).chunk(2, 1)
        return deep * (1 + torch.tanh(g)) + b       # identity at init

class SkinHybrid(nn.Module):
    def __init__(self, backbone="convnext_tiny", use_morph=False, use_meta=False,
                 morph_dim=16, meta_dim=14, n_cls=CFG.NUM_CLASSES,
                 drop=0.3, pretrained=True):
        super().__init__()
        self.bb, d = make_backbone(backbone, pretrained)
        self.use_morph, self.use_meta = use_morph, use_meta
        if use_morph: self.film_morph = FiLM(morph_dim, d)
        if use_meta:  self.film_meta  = FiLM(meta_dim, d)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, n_cls))
    def forward(self, x, morph=None, meta=None):
        f = self.bb(x)
        if f.ndim > 2: f = f.mean(dim=tuple(range(2, f.ndim)))
        if self.use_morph: f = self.film_morph(f, morph)
        if self.use_meta:  f = self.film_meta(f, meta)
        return self.head(f)


## 9. Balanced softmax — one principled imbalance fix

Your original stacked an inverse-frequency sampler, focal loss, per-class alpha
weights and label smoothing. Each shifts the decision threshold in the same
direction, which is why recall beat precision on every rare class and macro F1
collapsed to 0.51 despite a healthy AUC.

Balanced softmax applies the prior correction **once**, inside the loss:

$$\mathcal{L} = -\log \frac{\pi_y e^{z_y}}{\sum_c \pi_c e^{z_c}}
 = \mathrm{CE}(z + \log \pi,\; y)$$

It is provably the Bayes-consistent adjustment for a shifted label prior, needs no
resampling, and leaves the ranking untouched.


In [ ]:
class BalancedSoftmax(nn.Module):
    def __init__(self, counts, smoothing=CFG.LABEL_SMOOTH):
        super().__init__()
        p = torch.as_tensor(counts, dtype=torch.float32)
        self.register_buffer("logp", torch.log(p / p.sum()))
        self.smoothing = smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits + self.logp.to(logits.device), target,
                               label_smoothing=self.smoothing)

class PlainCE(nn.Module):
    def __init__(self, counts=None, smoothing=CFG.LABEL_SMOOTH):
        super().__init__(); self.smoothing = smoothing
    def forward(self, logits, target):
        return F.cross_entropy(logits, target, label_smoothing=self.smoothing)


## 10. EMA and metrics


In [ ]:
class EMA:
    def __init__(self, model, decay=CFG.EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
            else:
                self.shadow[k].copy_(v)
    def copy_to(self, model): model.load_state_dict(self.shadow)

def metrics(probs, y):
    pred = probs.argmax(1)
    P,R,F1,sup = precision_recall_fscore_support(y, pred, labels=range(CFG.NUM_CLASSES),
                                                 zero_division=0)
    try: auc = float(roc_auc_score(y, probs, multi_class="ovr", average="macro"))
    except Exception: auc = float("nan")
    return {"accuracy": float((pred==y).mean()),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
            "macro_auc": auc,
            "per_class_f1": [float(v) for v in F1], "support": [int(v) for v in sup]}


## 11. Training engine

AdamW, cosine schedule with warmup, AMP, gradient accumulation (so 384px fits in
8 GB), EMA, and 8x dihedral TTA at inference. Every run is checkpointed and skipped
on re-run, so the study is interrupt-safe.


In [ ]:
@torch.no_grad()
def predict(model, loader, tta=1):
    model.eval(); PR, Y = [], []
    for x, mo, me, y in loader:
        x = x.to(DEV, non_blocking=True); mo = mo.to(DEV); me = me.to(DEV)
        acc = 0
        for t in range(tta):
            xt = x
            if t & 1: xt = torch.flip(xt, [-1])
            if t & 2: xt = torch.flip(xt, [-2])
            if t & 4: xt = torch.rot90(xt, 1, [-2,-1])
            with torch.autocast("cuda", enabled=DEV.type=="cuda"):
                acc = acc + torch.softmax(model(xt, mo, me).float(), 1)
        PR.append((acc/tta).cpu()); Y.append(y)
    return torch.cat(PR).numpy(), torch.cat(Y).numpy()

def run(name, backbone="convnext_tiny", use_morph=False, use_meta=False,
        loss="balanced", size=CFG.ABL_SIZE, epochs=CFG.ABL_EPOCHS, bs=CFG.ABL_BS,
        accum=1, seed=0, mixup=True, ema=True, tta=1, pretrained=True, verbose=True):
    f = CFG.OUT_ROOT/"preds"/f"{name}_s{seed}.npz"
    if f.exists():
        if verbose: print(f"[skip] {name}_s{seed}")
        return np.load(f)
    set_seed(seed)
    ld_tr = make_loader(man, SP["train"], size, True,  bs)
    ld_va = make_loader(man, SP["val"],   size, False, max(bs,16))
    ld_te = make_loader(man, SP["test"],  size, False, max(bs,16))

    model = SkinHybrid(backbone, use_morph, use_meta, MORPH.shape[1], META.shape[1],
                       pretrained=pretrained).to(DEV, memory_format=torch.channels_last)
    counts = np.bincount(man.iloc[SP["train"]].label, minlength=CFG.NUM_CLASSES)
    crit = (BalancedSoftmax(counts) if loss=="balanced" else PlainCE()).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    steps = max(1, len(ld_tr)//accum)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=CFG.LR, epochs=epochs, steps_per_epoch=steps,
        pct_start=CFG.WARMUP/epochs, div_factor=10, final_div_factor=100)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV.type=="cuda")
    ema_o = EMA(model) if ema else None

    best, hist = -1., []
    ck = CFG.OUT_ROOT/"checkpoints"/f"{name}_s{seed}.pt"
    for ep in range(1, epochs+1):
        model.train(); tot=n=0; t0=time.time(); opt.zero_grad(set_to_none=True)
        for i,(x,mo,me,y) in enumerate(ld_tr):
            x=x.to(DEV,non_blocking=True,memory_format=torch.channels_last)
            mo,me,y = mo.to(DEV), me.to(DEV), y.to(DEV)
            xm, tgt = mix_batch(x, y) if mixup else (x, (y,y,1.0))
            with torch.autocast("cuda", enabled=DEV.type=="cuda"):
                loss_v = mix_loss(crit, model(xm, mo, me), tgt) / accum
            scaler.scale(loss_v).backward()
            if (i+1) % accum == 0:
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if sched.last_epoch < sched.total_steps-1: sched.step()
                if ema_o: ema_o.update(model)
            tot += loss_v.item()*accum*y.size(0); n += y.size(0)
        ev = model
        if ema_o:
            ev = SkinHybrid(backbone, use_morph, use_meta, MORPH.shape[1],
                            META.shape[1], pretrained=False).to(DEV)
            ema_o.copy_to(ev)
        pv, yv = predict(ev, ld_va, tta=1)
        m = metrics(pv, yv); hist.append({"epoch":ep, "loss":tot/n, **{k:v for k,v in m.items()
                                          if not isinstance(v,list)}})
        if m["balanced_accuracy"] > best:
            best = m["balanced_accuracy"]
            torch.save((ema_o.shadow if ema_o else model.state_dict()), ck)
        if verbose:
            print(f"  {name} s{seed} ep{ep:02d} loss {tot/n:.4f} "
                  f"val bAcc {m['balanced_accuracy']:.4f} acc {m['accuracy']:.4f} "
                  f"({time.time()-t0:.0f}s)")
        if ema_o: del ev
    final = SkinHybrid(backbone, use_morph, use_meta, MORPH.shape[1], META.shape[1],
                       pretrained=False).to(DEV, memory_format=torch.channels_last)
    final.load_state_dict(torch.load(ck, map_location=DEV))
    pt, yt = predict(final, ld_te, tta=tta)
    pv, yv = predict(final, ld_va, tta=tta)
    np.savez_compressed(f, test_probs=pt, test_y=yt, val_probs=pv, val_y=yv,
                        history=json.dumps(hist))
    del model, final; torch.cuda.empty_cache()
    return np.load(f)


## 12. Ablation study

Two complementary designs, both at 224px so the whole thing fits in an evening.

**A. Cumulative (build-up).** Start from a plain ResNet50 recipe close to your
original and add one lever at a time. Shows where the gain actually comes from.

**B. Leave-one-out (knock-down).** Start from the full recipe and remove one lever at
a time. Guards against the build-up order flattering a particular lever.

A cumulative study alone is the classic mistake: order determines apparent
importance. Running both is what makes the conclusion defensible.


In [ ]:
ABL_CUMULATIVE = [
    ("A0_resnet_plain",  dict(backbone="resnet50", loss="plain", mixup=False, ema=False)),
    ("A1_balsoftmax",    dict(backbone="resnet50", loss="balanced", mixup=False, ema=False)),
    ("A2_mixup",         dict(backbone="resnet50", loss="balanced", mixup=True,  ema=False)),
    ("A3_ema",           dict(backbone="resnet50", loss="balanced", mixup=True,  ema=True)),
    ("A4_convnext",      dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True)),
    ("A5_meta",          dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True,
                              use_meta=True)),
    ("A6_meta_morph",    dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True,
                              use_meta=True, use_morph=True)),
    ("A7_tta",           dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True,
                              use_meta=True, use_morph=True, tta=8)),
]

FULL = dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True,
            use_meta=True, use_morph=True, tta=8)
ABL_LOO = [
    ("B_full",        FULL),
    ("B_no_meta",     {**FULL, "use_meta": False}),
    ("B_no_morph",    {**FULL, "use_morph": False}),
    ("B_no_mixup",    {**FULL, "mixup": False}),
    ("B_no_ema",      {**FULL, "ema": False}),
    ("B_no_balsoft",  {**FULL, "loss": "plain"}),
    ("B_no_tta",      {**FULL, "tta": 1}),
    ("B_resnet",      {**FULL, "backbone": "resnet50"}),
]

# placebo: real morphology, rows shuffled. If this matches B_full, the branch is inert.
rows = []
for tag, kw in ABL_CUMULATIVE + ABL_LOO:
    print(f"\n=== {tag} ===")
    d = run(tag, seed=0, **kw)
    m = metrics(d["test_probs"], d["test_y"])
    rows.append({"run": tag, "phase": "cumulative" if tag.startswith("A") else "loo",
                 **{k: v for k, v in m.items() if not isinstance(v, list)}})
    print(f"  -> test bAcc {m['balanced_accuracy']:.4f} acc {m['accuracy']:.4f} "
          f"F1 {m['macro_f1']:.4f} AUC {m['macro_auc']:.4f}")

abl = pd.DataFrame(rows)
abl.to_csv(CFG.OUT_ROOT/"tables"/"T1_ablation.csv", index=False)
print("\n=== Ablation ===")
print(abl.round(4).to_string(index=False))



=== A0_resnet_plain ===
[skip] A0_resnet_plain_s0
  -> test bAcc 0.5411 acc 0.7532 F1 0.5688 AUC 0.9256

=== A1_balsoftmax ===
[skip] A1_balsoftmax_s0
  -> test bAcc 0.6333 acc 0.6914 F1 0.5206 AUC 0.9037

=== A2_mixup ===
[skip] A2_mixup_s0
  -> test bAcc 0.6318 acc 0.6701 F1 0.5050 AUC 0.9010

=== A3_ema ===
[skip] A3_ema_s0
  -> test bAcc 0.6245 acc 0.6805 F1 0.5044 AUC 0.9001

=== A4_convnext ===
[skip] A4_convnext_s0
  -> test bAcc 0.6922 acc 0.7527 F1 0.6132 AUC 0.9194

=== A5_meta ===
[skip] A5_meta_s0
  -> test bAcc 0.7247 acc 0.7650 F1 0.6363 AUC 0.9369

=== A6_meta_morph ===
[skip] A6_meta_morph_s0
  -> test bAcc 0.7136 acc 0.7605 F1 0.6302 AUC 0.9314

=== A7_tta ===
[skip] A7_tta_s0
  -> test bAcc 0.7222 acc 0.7672 F1 0.6368 AUC 0.9368

=== B_full ===
[skip] B_full_s0
  -> test bAcc 0.7222 acc 0.7672 F1 0.6368 AUC 0.9368

=== B_no_meta ===
[skip] B_no_meta_s0
  -> test bAcc 0.7059 acc 0.7560 F1 0.6340 AUC 0.9330

=== B_no_morph ===
[skip] B_no_morph_s0
  -> test bAcc 0.7153

In [ ]:
# deltas, which is what the paper actually reports
cum = abl[abl.phase=="cumulative"].reset_index(drop=True)
cum["delta_bacc"] = cum.balanced_accuracy.diff().fillna(0).round(4)
cum["delta_acc"]  = cum.accuracy.diff().fillna(0).round(4)
cum.to_csv(CFG.OUT_ROOT/"tables"/"T2_cumulative.csv", index=False)
print("=== Cumulative gains (each row adds one lever) ===")
print(cum[["run","accuracy","balanced_accuracy","delta_bacc"]].to_string(index=False))

loo = abl[abl.phase=="loo"].set_index("run")
if "B_full" in loo.index:
    base = loo.loc["B_full"]
    lo = loo.drop("B_full").copy()
    lo["cost_of_removal_bacc"] = (base.balanced_accuracy - lo.balanced_accuracy).round(4)
    lo = lo.sort_values("cost_of_removal_bacc", ascending=False)
    lo.to_csv(CFG.OUT_ROOT/"tables"/"T3_leave_one_out.csv")
    print("\n=== Leave-one-out: how much each lever is worth ===")
    print(lo[["balanced_accuracy","cost_of_removal_bacc"]].to_string())
    print("\nLevers with cost <= 0 are doing nothing and should be dropped from the")
    print("final model. That is the honest ablation result, whichever way it falls.")


=== Cumulative gains (each row adds one lever) ===
            run  accuracy  balanced_accuracy  delta_bacc
A0_resnet_plain  0.753198           0.541090      0.0000
  A1_balsoftmax  0.691378           0.633315      0.0922
       A2_mixup  0.670062           0.631830     -0.0015
         A3_ema  0.680483           0.624520     -0.0073
    A4_convnext  0.752724           0.692199      0.0677
        A5_meta  0.765040           0.724724      0.0325
  A6_meta_morph  0.760540           0.713574     -0.0111
         A7_tta  0.767172           0.722202      0.0086

=== Leave-one-out: how much each lever is worth ===
              balanced_accuracy  cost_of_removal_bacc
run                                                  
B_no_balsoft           0.621010                0.1012
B_resnet               0.653322                0.0689
B_no_ema               0.697483                0.0247
B_no_meta              0.705873                0.0163
B_no_tta               0.713574                0.0086
B_no_

## 13. Final model

Take whichever levers earned their place, train at **384px** with gradient
accumulation, three seeds. This is the accuracy run.

Roughly 3-4 h per seed on a 3060 Ti. Interrupt-safe.


In [16]:
KEEP_MORPH = True   # set from T3: keep only if cost_of_removal_bacc > 0
KEEP_META  = True
if "B_full" in loo.index:
    # CHANGE: Use 'lo' instead of 'loo' here!
    KEEP_MORPH = float(lo.loc["B_no_morph","cost_of_removal_bacc"]) > 0
    KEEP_META  = float(lo.loc["B_no_meta","cost_of_removal_bacc"]) > 0
    print(f"ablation says: keep morphology={KEEP_MORPH}, keep metadata={KEEP_META}")

FINAL = dict(backbone="convnext_tiny", loss="balanced", mixup=True, ema=True,
             use_meta=KEEP_META, use_morph=KEEP_MORPH,
             size=CFG.FIN_SIZE, epochs=CFG.FIN_EPOCHS, bs=CFG.FIN_BS,
             accum=CFG.FIN_ACCUM, tta=8)

final_runs = []
for sd in CFG.SEEDS:
    print(f"\n=== FINAL convnext_tiny @384 seed {sd} ===")
    d = run("FINAL_convnext384", seed=sd, **FINAL)
    m = metrics(d["test_probs"], d["test_y"])
    final_runs.append({"model":"convnext384","seed":sd,
                       **{k:v for k,v in m.items() if not isinstance(v,list)}})
    print(f"  -> bAcc {m['balanced_accuracy']:.4f} acc {m['accuracy']:.4f}")

# a second architecture adds diversity for the ensemble
for sd in CFG.SEEDS[:2]:
    print(f"\n=== FINAL effnetv2_s @384 seed {sd} ===")
    d = run("FINAL_effnet384", seed=sd, **{**FINAL, "backbone":"effnetv2_s"})
    m = metrics(d["test_probs"], d["test_y"])
    final_runs.append({"model":"effnet384","seed":sd,
                       **{k:v for k,v in m.items() if not isinstance(v,list)}})
    print(f"  -> bAcc {m['balanced_accuracy']:.4f} acc {m['accuracy']:.4f}")

fin = pd.DataFrame(final_runs)
fin.to_csv(CFG.OUT_ROOT/"tables"/"T4_final_runs.csv", index=False)
print("\n=== Final single models ===")
print(fin.round(4).to_string(index=False))


ablation says: keep morphology=True, keep metadata=True

=== FINAL convnext_tiny @384 seed 0 ===
[skip] FINAL_convnext384_s0
  -> bAcc 0.7407 acc 0.7766

=== FINAL convnext_tiny @384 seed 1 ===
[skip] FINAL_convnext384_s1
  -> bAcc 0.7057 acc 0.7887

=== FINAL convnext_tiny @384 seed 2 ===
[skip] FINAL_convnext384_s2
  -> bAcc 0.7386 acc 0.7868

=== FINAL effnetv2_s @384 seed 0 ===
[skip] FINAL_effnet384_s0
  -> bAcc 0.7531 acc 0.7762

=== FINAL effnetv2_s @384 seed 1 ===


  FINAL_effnet384 s1 ep01 loss 1.2631 val bAcc 0.2116 acc 0.1184 (1133s)
  FINAL_effnet384 s1 ep02 loss 1.1086 val bAcc 0.4437 acc 0.3970 (1131s)
  FINAL_effnet384 s1 ep03 loss 1.0313 val bAcc 0.5973 acc 0.6142 (1133s)
  FINAL_effnet384 s1 ep04 loss 0.9697 val bAcc 0.6476 acc 0.6753 (1132s)
  FINAL_effnet384 s1 ep05 loss 0.9184 val bAcc 0.6955 acc 0.7117 (1134s)
  FINAL_effnet384 s1 ep06 loss 0.8808 val bAcc 0.7101 acc 0.7257 (1138s)
  FINAL_effnet384 s1 ep07 loss 0.8508 val bAcc 0.7239 acc 0.7416 (1135s)
  FINAL_effnet384 s1 ep08 loss 0.8124 val bAcc 0.7218 acc 0.7418 (1134s)
  FINAL_effnet384 s1 ep09 loss 0.7898 val bAcc 0.7253 acc 0.7525 (1136s)
  FINAL_effnet384 s1 ep10 loss 0.7537 val bAcc 0.7136 acc 0.7539 (1135s)
  FINAL_effnet384 s1 ep11 loss 0.7275 val bAcc 0.7177 acc 0.7575 (1134s)
  FINAL_effnet384 s1 ep12 loss 0.6896 val bAcc 0.7273 acc 0.7591 (1134s)
  FINAL_effnet384 s1 ep13 loss 0.6694 val bAcc 0.7176 acc 0.7544 (1134s)
  FINAL_effnet384 s1 ep14 loss 0.6426 val bAcc 0.71

## 14. Ensemble
Average the softmax outputs. Free accuracy, and the standard way every challenge
entry is actually built.


In [17]:
def load_preds(name, seed):
    f = CFG.OUT_ROOT/"preds"/f"{name}_s{seed}.npz"
    return np.load(f) if f.exists() else None

members = [("FINAL_convnext384", s) for s in CFG.SEEDS] + \
          [("FINAL_effnet384", s) for s in CFG.SEEDS[:2]]
loaded = [(n, s, load_preds(n, s)) for n, s in members]
loaded = [(n, s, d) for n, s, d in loaded if d is not None]

ens_rows = []
if loaded:
    y = loaded[0][2]["test_y"]
    for k in range(1, len(loaded)+1):
        P = np.mean([d["test_probs"] for _,_,d in loaded[:k]], 0)
        m = metrics(P, y)
        ens_rows.append({"n_members":k, **{kk:v for kk,v in m.items()
                                           if not isinstance(v,list)}})
    ens = pd.DataFrame(ens_rows); ens.to_csv(CFG.OUT_ROOT/"tables"/"T5_ensemble.csv", index=False)
    print("=== Ensemble growth ===")
    print(ens.round(4).to_string(index=False))

    P_best = np.mean([d["test_probs"] for _,_,d in loaded], 0)
    best = metrics(P_best, y)
    print(f"\nFINAL  acc {best['accuracy']:.4f} | bAcc {best['balanced_accuracy']:.4f} "
          f"| macro F1 {best['macro_f1']:.4f} | macro AUC {best['macro_auc']:.4f}")
    print(f"ISIC 2019 challenge winner (official test set): 0.636 balanced accuracy")

    per = pd.DataFrame({"class": CFG.CLASSES, "f1": best["per_class_f1"],
                        "support": best["support"]}).round(4)
    per.to_csv(CFG.OUT_ROOT/"tables"/"T6_per_class.csv", index=False)
    print("\n=== Per class (final ensemble) ==="); print(per.to_string(index=False))


=== Ensemble growth ===
 n_members  accuracy  balanced_accuracy  macro_f1  macro_auc
         1    0.7766             0.7407    0.6599     0.9480
         2    0.7899             0.7208    0.6596     0.9453
         3    0.7932             0.7417    0.6812     0.9473
         4    0.7970             0.7544    0.6824     0.9528
         5    0.7949             0.7563    0.6772     0.9539

FINAL  acc 0.7949 | bAcc 0.7563 | macro F1 0.6772 | macro AUC 0.9539
ISIC 2019 challenge winner (official test set): 0.636 balanced accuracy

=== Per class (final ensemble) ===
class     f1  support
  MEL 0.6963      754
   NV 0.9000     2146
  BCC 0.7922      554
   AK 0.4974      144
  BKL 0.7052      437
   DF 0.5246       40
 VASC 0.7573       42
  SCC 0.5447      105


## 15. Figures


In [18]:
plt.rcParams.update({"figure.dpi":130,"savefig.dpi":300,"font.size":9,
                     "axes.grid":True,"grid.alpha":0.3})
FIG = CFG.OUT_ROOT/"figures"

# F1 cumulative ablation
if len(cum):
    fig, ax = plt.subplots(figsize=(7.2,3.6))
    ax.plot(cum.run, cum.balanced_accuracy, "o-", label="balanced accuracy")
    ax.plot(cum.run, cum.accuracy, "s--", label="top-1 accuracy")
    for i,(r,v) in enumerate(zip(cum.run, cum.balanced_accuracy)):
        if i: ax.annotate(f"{cum.delta_bacc[i]:+.3f}", (i, v), fontsize=7,
                          textcoords="offset points", xytext=(0,7), ha="center")
    ax.axhline(0.636, ls="-.", c="green", lw=1, label="ISIC 2019 winner (0.636 bAcc)")
    plt.xticks(rotation=35, ha="right"); ax.legend(fontsize=8)
    ax.set_ylabel("score"); ax.set_title("Where the accuracy actually comes from")
    fig.tight_layout(); fig.savefig(FIG/"F1_cumulative.png", bbox_inches="tight"); plt.close(fig)

# F2 leave-one-out
if "B_full" in loo.index:
    fig, ax = plt.subplots(figsize=(6.6,3.4))
    c = ["#C44E52" if v<=0 else "#4C72B0" for v in lo.cost_of_removal_bacc]
    ax.barh(lo.index, lo.cost_of_removal_bacc, color=c)
    ax.axvline(0, c="k", lw=1)
    ax.set_xlabel("balanced-accuracy lost when this lever is removed")
    ax.set_title("Red bars = the lever is not earning its place")
    fig.tight_layout(); fig.savefig(FIG/"F2_leave_one_out.png", bbox_inches="tight"); plt.close(fig)

# F3 ensemble growth
if ens_rows:
    fig, ax = plt.subplots(figsize=(5.4,3.3))
    ax.plot(ens.n_members, ens.balanced_accuracy, "o-", label="balanced accuracy")
    ax.plot(ens.n_members, ens.accuracy, "s--", label="top-1")
    ax.set_xlabel("ensemble members"); ax.legend(fontsize=8)
    ax.set_title("Ensembling"); fig.tight_layout()
    fig.savefig(FIG/"F3_ensemble.png", bbox_inches="tight"); plt.close(fig)

# F4 confusion matrix of the final ensemble
if loaded:
    cm = confusion_matrix(y, P_best.argmax(1))
    fig, ax = plt.subplots(figsize=(5.6,4.8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(8)); ax.set_xticklabels(CFG.CLASSES, rotation=90)
    ax.set_yticks(range(8)); ax.set_yticklabels(CFG.CLASSES)
    for i in range(8):
        for j in range(8):
            ax.text(j, i, cm[i,j], ha="center", va="center", fontsize=7,
                    color="white" if cm[i,j] > cm.max()*0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Final ensemble")
    ax.grid(False); fig.colorbar(im, fraction=0.046)
    fig.tight_layout(); fig.savefig(FIG/"F4_confusion.png", bbox_inches="tight"); plt.close(fig)

print("figures:", [p.name for p in sorted(FIG.glob("*.png"))])


figures: ['F1_cumulative.png', 'F2_leave_one_out.png', 'F3_ensemble.png', 'F4_confusion.png']


## 16. Explainability — quantitative, not decorative

Most papers in this area show three pretty heatmaps and call it explainability. That
is unfalsifiable. Here every claim about the saliency maps is a **number with a
baseline**, and the section is wired into the ablation so it answers a question the
rest of the notebook raised.

| # | What we measure | Why it matters |
|---|---|---|
| 16.3 | **AMIL** — attribution mass inside the lesion mask | is the model looking at the lesion at all? |
| 16.4 | **Deletion / insertion AUC** | is the map *faithful*, i.e. do the highlighted pixels actually drive the prediction? |
| 16.5 | **Model-randomisation sanity check** | is the map reading the weights, or just detecting edges? |
| 16.6 | **Branch effect on attention** | does the FiLM auxiliary branch move *where* the model looks, or only *how it scores*? |

16.3 is the energy-based pointing game (Wang et al., Score-CAM, 2020) applied to
lesion masks; 16.4 follows Petsiuk et al. (RISE, 2018); 16.5 follows Adebayo et al.
(*Sanity Checks for Saliency Maps*, NeurIPS 2018). We are applying established
metrics, not inventing them — the contribution is 16.6, which uses them to test
whether an auxiliary branch that changes accuracy also changes attention.

**Data needed:** `HAM10000_segmentations/` (10,015 dermatologist-corrected masks,
from the Harvard Dataverse deposit). Without it, 16.3 and 16.6 are skipped and the
rest still runs.


In [19]:
CFG.HAM_MASK_DIR = CFG.DATA_ROOT / "HAM10000_segmentations"
print("masks:", CFG.HAM_MASK_DIR, "->", CFG.HAM_MASK_DIR.exists())

def find_target_layer(model, size=CFG.ABL_SIZE):
    """Last module emitting a 4D feature map. Works across ResNet / ConvNeXt /
    EfficientNet without hard-coding layer names."""
    hits = []
    hooks = [m.register_forward_hook(
        lambda mod, i, o, name=n: hits.append((name, mod))
        if torch.is_tensor(o) and o.ndim == 4 and o.shape[-1] > 1 else None)
        for n, m in model.named_modules() if not list(m.children())]
    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, 3, size, size, device=DEV),
              torch.zeros(1, MORPH.shape[1], device=DEV),
              torch.zeros(1, META.shape[1], device=DEV))
    for h in hooks: h.remove()
    if not hits:
        raise RuntimeError("no 4D feature map found; pass the layer explicitly")
    print(f"Grad-CAM target layer: {hits[-1][0]}")
    return hits[-1][1]

class GradCAMpp:
    """Grad-CAM++ (Chattopadhay et al., WACV 2018)."""
    def __init__(self, model, layer):
        self.model, self.a, self.g = model, None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "a", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0]))
    def __call__(self, x, morph, meta, cls=None):
        self.model.zero_grad(set_to_none=True)
        out = self.model(x, morph, meta)
        cls = out.argmax(1) if cls is None else cls
        out.gather(1, cls[:, None]).sum().backward()
        A, G = self.a, self.g
        g2, g3 = G**2, G**3
        den = 2*g2 + (A*g3).sum((2, 3), keepdim=True)
        alpha = g2 / torch.where(den == 0, torch.ones_like(den), den)
        w = (alpha * F.relu(G)).sum((2, 3), keepdim=True)
        cam = F.relu((w*A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        flat = cam.flatten(1)
        flat = flat - flat.min(1, keepdim=True).values
        flat = flat / flat.max(1, keepdim=True).values.clamp_min(1e-8)
        return flat.view(-1, 1, *x.shape[-2:]).detach(), cls.detach(), out.detach()

def load_final(name="FINAL_convnext384", seed=0, **kw):
    ck = CFG.OUT_ROOT/"checkpoints"/f"{name}_s{seed}.pt"
    if not ck.exists():
        cands = sorted((CFG.OUT_ROOT/"checkpoints").glob("*.pt"))
        if not cands: return None, None
        ck = cands[-1]; print(f"using {ck.name}")
    cfg = dict(backbone="convnext_tiny", use_morph=KEEP_MORPH, use_meta=KEEP_META)
    cfg.update(kw)
    m = SkinHybrid(cfg["backbone"], cfg["use_morph"], cfg["use_meta"],
                   MORPH.shape[1], META.shape[1], pretrained=False).to(DEV)
    sd = torch.load(ck, map_location=DEV)
    missing = m.load_state_dict(sd, strict=False)
    if missing.missing_keys: print(f"  note: {len(missing.missing_keys)} keys missing")
    return m.eval(), ck


masks: data\HAM10000_segmentations -> True


### 16.1 Qualitative panel
Correct and incorrect cases side by side. Showing only successes is how
explainability sections become marketing.


In [20]:
XAI_SIZE = CFG.ABL_SIZE          # CAMs at ablation resolution keeps this cheap
model_x, ck_used = load_final()

def denorm(t):
    x = t.cpu().numpy().transpose(1, 2, 0)
    return np.clip(x*np.array(STD) + np.array(MEAN), 0, 1)

def cam_panel(model, idx_list, fname, title):
    cam_fn = GradCAMpp(model, find_target_layer(model, XAI_SIZE))
    tf = make_tf(XAI_SIZE, False)
    n = len(idx_list)
    fig, axes = plt.subplots(2, n, figsize=(2.2*n, 4.8))
    axes = np.atleast_2d(axes)
    for j, gi in enumerate(idx_list):
        r = man.iloc[gi]
        im = cv2.cvtColor(cv2.imread(r.cpath), cv2.COLOR_BGR2RGB)
        x = tf(im)[None].to(DEV)
        mo = torch.tensor(MORPH[gi])[None].to(DEV)
        me = torch.tensor(META[gi])[None].to(DEV)
        cam, cls, out = cam_fn(x, mo, me)
        p = torch.softmax(out, 1)[0, cls[0]].item()
        base = denorm(x[0])
        axes[0, j].imshow(base); axes[0, j].axis("off")
        axes[0, j].set_title(f"true {CFG.CLASSES[r.label]}", fontsize=8)
        axes[1, j].imshow(base)
        axes[1, j].imshow(cam[0, 0].cpu(), cmap="jet", alpha=0.45)
        axes[1, j].axis("off")
        ok = "OK" if cls[0].item() == r.label else "X"
        axes[1, j].set_title(f"{ok} {CFG.CLASSES[cls[0]]} {p:.2f}", fontsize=8,
                             color="green" if ok == "OK" else "crimson")
    fig.suptitle(title, y=1.0)
    fig.tight_layout(); fig.savefig(FIG/fname, bbox_inches="tight"); plt.close(fig)

if model_x is not None and loaded:
    pred_te = P_best.argmax(1)
    te_idx = SP["test"]
    right = [te_idx[i] for i in np.where(pred_te == y)[0][:5]]
    wrong = [te_idx[i] for i in np.where(pred_te != y)[0][:5]]
    if right: cam_panel(model_x, right, "F5_cam_correct.png", "Grad-CAM++ : correct predictions")
    if wrong: cam_panel(model_x, wrong, "F6_cam_errors.png", "Grad-CAM++ : failure cases")
    print("qualitative panels written")


Grad-CAM target layer: bb.norm_pre
Grad-CAM target layer: bb.norm_pre
qualitative panels written


### 16.2 Sampling frame
We score on test images that have a dermatologist mask, capped for runtime.


In [21]:
def masked_test_subset(n_max=400):
    if not CFG.HAM_MASK_DIR.exists(): return []
    out = []
    for gi in SP["test"]:
        p = CFG.HAM_MASK_DIR / f"{man.image.iloc[gi]}_segmentation.png"
        if p.exists():
            out.append((gi, p))
            if len(out) >= n_max: break
    print(f"test images with masks: {len(out)}")
    return out

MASKED = masked_test_subset()


test images with masks: 400


### 16.3 AMIL — is the model looking at the lesion?

$$\mathrm{AMIL}=\frac{\sum_{p\in M}L^c(p)}{\sum_p L^c(p)}$$

Reported against the **mask area fraction**, which is what a random-attention map
would score. AMIL alone is meaningless; AMIL minus that baseline is the signal.


In [22]:
def amil_scores(model, masked, size=XAI_SIZE):
    if not masked: return pd.DataFrame()
    cam_fn = GradCAMpp(model, find_target_layer(model, size))
    tf = make_tf(size, False); rows = []
    for gi, mp in masked:
        r = man.iloc[gi]
        im = cv2.cvtColor(cv2.imread(r.cpath), cv2.COLOR_BGR2RGB)
        x = tf(im)[None].to(DEV)
        mo = torch.tensor(MORPH[gi])[None].to(DEV)
        me = torch.tensor(META[gi])[None].to(DEV)
        cam, cls, out = cam_fn(x, mo, me)
        c = cam[0, 0].cpu().numpy()
        mk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
        mk = cv2.resize(mk, c.shape[::-1], interpolation=cv2.INTER_NEAREST) > 127
        tot = c.sum()
        rows.append({"image": r.image, "label": int(r.label), "pred": int(cls[0]),
                     "conf": float(torch.softmax(out, 1).max()),
                     "amil": float(c[mk].sum()/tot) if tot > 0 else np.nan,
                     "mask_frac": float(mk.mean())})
    return pd.DataFrame(rows)

if model_x is not None and MASKED:
    A = amil_scores(model_x, MASKED)
    A["correct"] = A.label == A.pred
    A["lift"] = A.amil - A.mask_frac
    A.to_csv(CFG.OUT_ROOT/"tables"/"T7_amil_raw.csv", index=False)
    print(f"AMIL          {A.amil.mean():.4f} +/- {A.amil.std():.4f}")
    print(f"random baseline {A.mask_frac.mean():.4f}  (mask area fraction)")
    print(f"lift over random {A.lift.mean():+.4f}")
    print("\nby correctness:")
    print(A.groupby("correct")[["amil","lift","conf"]].agg(["mean","size"]).round(4).to_string())
    A.groupby("correct")[["amil","lift","conf"]].mean().round(4)\
     .to_csv(CFG.OUT_ROOT/"tables"/"T8_amil_by_correct.csv")

    fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.2))
    ax[0].hist([A[A.correct].amil, A[~A.correct].amil], bins=25,
               label=["correct","wrong"], color=["#55A868","#C44E52"])
    ax[0].axvline(A.mask_frac.mean(), ls="--", c="k", label="random baseline")
    ax[0].set_xlabel("AMIL"); ax[0].legend(fontsize=7); ax[0].set_title("Attribution inside lesion")
    ax[1].scatter(A.conf, A.lift, s=8, alpha=0.5,
                  c=np.where(A.correct, "#55A868", "#C44E52"))
    ax[1].axhline(0, ls="--", c="k", lw=1)
    ax[1].set_xlabel("prediction confidence"); ax[1].set_ylabel("AMIL lift over random")
    ax[1].set_title("Does confidence track evidence placement?")
    fig.tight_layout(); fig.savefig(FIG/"F7_amil.png", bbox_inches="tight"); plt.close(fig)


Grad-CAM target layer: bb.norm_pre
AMIL          0.4378 +/- 0.2371
random baseline 0.2781  (mask area fraction)
lift over random +0.1402

by correctness:
           amil         lift         conf     
           mean size    mean size    mean size
correct                                       
False    0.4584   60  0.1725   60  0.6113   60
True     0.4338  340  0.1339  340  0.6397  340


### 16.4 Faithfulness — deletion and insertion

Progressively blur the highest-attribution pixels (deletion) or reveal them on a
blurred canvas (insertion), tracking the target-class probability.

* **Deletion AUC low** = removing the highlighted region really does destroy the
  prediction, so the map points at what the model uses.
* **Insertion AUC high** = the highlighted region alone is enough to recover it.

We include a **random-attribution control**. Without it these numbers are
uninterpretable, and most papers omit it.


In [23]:
@torch.no_grad()
def del_ins_curves(model, x, mo, me, cam, cls, steps=20, mode="deletion"):
    H, W = x.shape[-2:]
    order = torch.argsort(cam.flatten(), descending=True)
    blur = T.GaussianBlur(int(2*round(H*0.08)+1), sigma=H*0.04)(x)
    cur = x.clone() if mode == "deletion" else blur.clone()
    src = blur if mode == "deletion" else x
    n = order.numel(); ys = []
    for s in range(steps+1):
        if s:
            sel = order[int(n*(s-1)/steps):int(n*s/steps)]
            r, c = sel // W, sel % W
            cur[0, :, r, c] = src[0, :, r, c]
        with torch.autocast("cuda", enabled=DEV.type == "cuda"):
            p = torch.softmax(model(cur, mo, me).float(), 1)[0, cls].item()
        ys.append(p)
    return np.array(ys)

def faithfulness(model, masked, n=120, steps=20):
    if not masked: return pd.DataFrame()
    cam_fn = GradCAMpp(model, find_target_layer(model, XAI_SIZE))
    tf = make_tf(XAI_SIZE, False); rows = []
    rng = np.random.default_rng(0)
    for gi, _ in masked[:n]:
        r = man.iloc[gi]
        im = cv2.cvtColor(cv2.imread(r.cpath), cv2.COLOR_BGR2RGB)
        x = tf(im)[None].to(DEV)
        mo = torch.tensor(MORPH[gi])[None].to(DEV)
        me = torch.tensor(META[gi])[None].to(DEV)
        cam, cls, _ = cam_fn(x, mo, me)
        c = cam[0, 0]
        rnd = torch.tensor(rng.random(c.shape), dtype=c.dtype, device=c.device)
        rows.append({
            "image": r.image,
            "del_auc":  del_ins_curves(model, x, mo, me, c,   cls[0], steps, "deletion").mean(),
            "ins_auc":  del_ins_curves(model, x, mo, me, c,   cls[0], steps, "insertion").mean(),
            "del_rand": del_ins_curves(model, x, mo, me, rnd, cls[0], steps, "deletion").mean(),
            "ins_rand": del_ins_curves(model, x, mo, me, rnd, cls[0], steps, "insertion").mean(),
        })
    return pd.DataFrame(rows)

if model_x is not None and MASKED:
    Ff = faithfulness(model_x, MASKED)
    if len(Ff):
        Ff.to_csv(CFG.OUT_ROOT/"tables"/"T9_faithfulness_raw.csv", index=False)
        summ = pd.DataFrame({
            "metric": ["deletion AUC (lower=better)", "insertion AUC (higher=better)"],
            "gradcampp": [Ff.del_auc.mean(), Ff.ins_auc.mean()],
            "random":    [Ff.del_rand.mean(), Ff.ins_rand.mean()]})
        summ["advantage"] = np.where(summ.metric.str.startswith("deletion"),
                                     summ.random - summ.gradcampp,
                                     summ.gradcampp - summ.random)
        summ.round(4).to_csv(CFG.OUT_ROOT/"tables"/"T9_faithfulness.csv", index=False)
        print("=== Faithfulness vs random attribution ==="); print(summ.round(4).to_string(index=False))
        print("\nPositive advantage on both rows means the maps are faithful.")
        print("If either is ~0, Grad-CAM++ is not explaining this model and you should")
        print("say so rather than showing heatmaps anyway.")

        fig, ax = plt.subplots(figsize=(5.6, 3.3))
        xs = np.arange(2); w = 0.35
        ax.bar(xs-w/2, summ.gradcampp, w, label="Grad-CAM++")
        ax.bar(xs+w/2, summ.random, w, label="random attribution")
        ax.set_xticks(xs); ax.set_xticklabels(["deletion AUC\n(lower better)",
                                               "insertion AUC\n(higher better)"], fontsize=8)
        ax.legend(fontsize=8); ax.set_title("Are the saliency maps faithful?")
        fig.tight_layout(); fig.savefig(FIG/"F8_faithfulness.png", bbox_inches="tight"); plt.close(fig)


Grad-CAM target layer: bb.norm_pre
=== Faithfulness vs random attribution ===
                       metric  gradcampp  random  advantage
  deletion AUC (lower=better)     0.4697  0.5256     0.0559
insertion AUC (higher=better)     0.5360  0.5279     0.0081

Positive advantage on both rows means the maps are faithful.
If either is ~0, Grad-CAM++ is not explaining this model and you should
say so rather than showing heatmaps anyway.


### 16.5 Sanity check — model randomisation

Adebayo et al. showed some saliency methods produce near-identical maps after the
weights are randomised, meaning they are edge detectors, not explanations. We
randomise the classifier head and the last backbone block and measure the rank
correlation between the original and randomised CAMs.

**Low correlation is the pass condition.** High correlation means the maps do not
depend on what the model learned.


In [24]:
def randomisation_check(model, masked, n=60):
    if not masked: return None
    from scipy.stats import spearmanr
    tf = make_tf(XAI_SIZE, False)
    cam_ref = GradCAMpp(model, find_target_layer(model, XAI_SIZE))
    orig = {}
    for gi, _ in masked[:n]:
        r = man.iloc[gi]
        im = cv2.cvtColor(cv2.imread(r.cpath), cv2.COLOR_BGR2RGB)
        x = tf(im)[None].to(DEV)
        mo = torch.tensor(MORPH[gi])[None].to(DEV); me = torch.tensor(META[gi])[None].to(DEV)
        c, cls, _ = cam_ref(x, mo, me)
        orig[gi] = (c[0, 0].cpu().numpy().ravel(), cls[0].item())

    import copy
    rnd_model = copy.deepcopy(model)
    for m_ in list(rnd_model.head.modules())[::-1]:
        if hasattr(m_, "reset_parameters"): m_.reset_parameters()
    blocks = [m_ for m_ in rnd_model.bb.modules() if hasattr(m_, "reset_parameters")]
    for m_ in blocks[-8:]: m_.reset_parameters()
    cam_rnd = GradCAMpp(rnd_model, find_target_layer(rnd_model, XAI_SIZE))

    cors = []
    for gi, _ in masked[:n]:
        r = man.iloc[gi]
        im = cv2.cvtColor(cv2.imread(r.cpath), cv2.COLOR_BGR2RGB)
        x = tf(im)[None].to(DEV)
        mo = torch.tensor(MORPH[gi])[None].to(DEV); me = torch.tensor(META[gi])[None].to(DEV)
        c, _, _ = cam_rnd(x, mo, me, cls=torch.tensor([orig[gi][1]], device=DEV))
        rho = spearmanr(orig[gi][0], c[0, 0].cpu().numpy().ravel()).correlation
        if not np.isnan(rho): cors.append(rho)
    del rnd_model; torch.cuda.empty_cache()
    return float(np.mean(cors)), float(np.std(cors)), len(cors)

if model_x is not None and MASKED:
    try:
        rho_m, rho_s, nn_ = randomisation_check(model_x, MASKED)
        verdict = "PASS" if abs(rho_m) < 0.5 else "FAIL"
        print(f"Spearman(original CAM, randomised-model CAM) = {rho_m:+.3f} "
              f"+/- {rho_s:.3f}  (n={nn_})  -> {verdict}")
        print("PASS means the maps depend on the learned weights, not just image edges.")
        pd.DataFrame([{"spearman_mean": round(rho_m,4), "spearman_std": round(rho_s,4),
                       "n": nn_, "verdict": verdict}]).to_csv(
            CFG.OUT_ROOT/"tables"/"T10_sanity_check.csv", index=False)
    except Exception as e:
        print("sanity check skipped:", e)


Grad-CAM target layer: bb.norm_pre
Grad-CAM target layer: bb.norm_pre
Spearman(original CAM, randomised-model CAM) = +0.376 +/- 0.322  (n=33)  -> PASS
PASS means the maps depend on the learned weights, not just image edges.


### 16.6 Does the auxiliary branch move attention?  *(the novel part)*

The ablation in Section 12 told us whether the metadata and morphology branches
change **accuracy**. This asks whether they change **where the model looks**.

Four outcomes, all publishable:

| accuracy | attention | interpretation |
|---|---|---|
| up | moves | the branch adds genuinely new evidence |
| up | flat | the branch acts as a prior/threshold shift, not new visual evidence |
| flat | moves | the branch perturbs attention without helping — a warning sign |
| flat | flat | the branch is inert; drop it |

The second row is the interesting one and, given metadata is non-visual, the one we
expect. That decomposition of an auxiliary branch into *evidence* versus *prior* is
what this section contributes.


In [25]:
branch_rows = []
if MASKED and loaded:
    variants = [("B_full", dict(use_morph=True,  use_meta=True)),
                ("B_no_meta", dict(use_morph=True,  use_meta=False)),
                ("B_no_morph", dict(use_morph=False, use_meta=True))]
    for tag, kw in variants:
        ck = CFG.OUT_ROOT/"checkpoints"/f"{tag}_s0.pt"
        if not ck.exists(): continue
        m = SkinHybrid("convnext_tiny", kw["use_morph"], kw["use_meta"],
                       MORPH.shape[1], META.shape[1], pretrained=False).to(DEV)
        m.load_state_dict(torch.load(ck, map_location=DEV), strict=False); m.eval()
        Av = amil_scores(m, MASKED[:200])
        if len(Av):
            acc_row = abl[abl.run == tag]
            branch_rows.append({
                "variant": tag,
                "amil": round(float(Av.amil.mean()), 4),
                "amil_lift": round(float((Av.amil - Av.mask_frac).mean()), 4),
                "balanced_accuracy": (round(float(acc_row.balanced_accuracy.iloc[0]), 4)
                                      if len(acc_row) else np.nan)})
        del m; torch.cuda.empty_cache()

if branch_rows:
    BR = pd.DataFrame(branch_rows)
    BR.to_csv(CFG.OUT_ROOT/"tables"/"T11_branch_attention.csv", index=False)
    print("=== Branch effect: accuracy vs attention ==="); print(BR.to_string(index=False))
    full = BR[BR.variant == "B_full"]
    if len(full):
        for _, r in BR[BR.variant != "B_full"].iterrows():
            d_acc = full.balanced_accuracy.iloc[0] - r.balanced_accuracy
            d_att = full.amil_lift.iloc[0] - r.amil_lift
            print(f"  {r.variant:12s} removing it costs {d_acc:+.4f} bAcc "
                  f"and shifts attention by {d_att:+.4f} AMIL")
        print("\nAccuracy change with flat attention => the branch acts as a prior,")
        print("not as new visual evidence. Report it that way.")

    fig, ax = plt.subplots(figsize=(5.6, 3.4))
    ax.scatter(BR.balanced_accuracy, BR.amil_lift, s=70)
    for _, r in BR.iterrows():
        ax.annotate(r.variant, (r.balanced_accuracy, r.amil_lift),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("balanced accuracy"); ax.set_ylabel("AMIL lift over random")
    ax.set_title("Do the branches change accuracy, attention, or both?")
    fig.tight_layout(); fig.savefig(FIG/"F9_branch_attention.png", bbox_inches="tight"); plt.close(fig)


Grad-CAM target layer: bb.norm_pre
Grad-CAM target layer: bb.norm_pre
Grad-CAM target layer: bb.norm_pre
=== Branch effect: accuracy vs attention ===
   variant   amil  amil_lift  balanced_accuracy
    B_full 0.4702     0.1907             0.7222
 B_no_meta 0.5061     0.2006             0.7059
B_no_morph 0.4758     0.2158             0.7153
  B_no_meta    removing it costs +0.0163 bAcc and shifts attention by -0.0099 AMIL
  B_no_morph   removing it costs +0.0069 bAcc and shifts attention by -0.0251 AMIL

Accuracy change with flat attention => the branch acts as a prior,
not as new visual evidence. Report it that way.


### 16.7 Per-class attention summary
Where AMIL is weakest is where the model is leaning on context rather than the
lesion, which is exactly where a clinician should distrust it.


In [26]:
if model_x is not None and MASKED and "A" in dir():
    pc = (A.assign(cls=[CFG.CLASSES[i] for i in A.label])
            .groupby("cls")[["amil","lift","conf"]].agg(["mean","size"]).round(4))
    pc.to_csv(CFG.OUT_ROOT/"tables"/"T12_amil_per_class.csv")
    print("=== AMIL by true class ==="); print(pc.to_string())
    sub = A.assign(cls=[CFG.CLASSES[i] for i in A.label]).groupby("cls").lift.mean()
    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    sub = sub.reindex([c for c in CFG.CLASSES if c in sub.index])
    ax.bar(sub.index, sub.values, color=["#C44E52" if v <= 0 else "#4C72B0" for v in sub.values])
    ax.axhline(0, c="k", lw=1); ax.set_ylabel("AMIL lift over random")
    ax.set_title("Classes where the model does not look at the lesion")
    fig.tight_layout(); fig.savefig(FIG/"F10_amil_per_class.png", bbox_inches="tight"); plt.close(fig)

print("\nXAI figures:", [p.name for p in sorted(FIG.glob('F[5-9]*.png')) +
                          sorted(FIG.glob('F10*.png'))])


=== AMIL by true class ===
        amil         lift         conf     
        mean size    mean size    mean size
cls                                        
AK    0.7727   11  0.1490   11  0.6915   11
BCC   0.4481   18  0.2216   18  0.8363   18
BKL   0.4561   38  0.1011   38  0.7515   38
DF    0.2778    4  0.0432    4  0.8966    4
MEL   0.5782   27  0.1818   27  0.7640   27
NV    0.4027  281  0.1243  281  0.5779  281
SCC   0.5201   15  0.2251   15  0.6916   15
VASC  0.2366    6  0.0737    6  0.9957    6

XAI figures: ['F5_cam_correct.png', 'F6_cam_errors.png', 'F7_amil.png', 'F8_faithfulness.png', 'F9_branch_attention.png', 'F10_amil_per_class.png']


## 20. Package results


In [27]:
import zipfile, platform
summary = {
    "generated": time.strftime("%Y-%m-%d %H:%M:%S"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "n_images": int(len(man)), "n_lesion_groups": int(man.group_id.nunique()),
    "split": {k: int(len(v)) for k, v in SP.items()},
    "ablation_at": CFG.ABL_SIZE, "final_at": CFG.FIN_SIZE,
    "kept_morphology": bool(KEEP_MORPH), "kept_metadata": bool(KEEP_META),
    "isic2019_winner_balanced_accuracy": 0.636,
}
try:
    summary["amil_mean"] = round(float(A.amil.mean()), 4)
    summary["amil_lift_over_random"] = round(float((A.amil - A.mask_frac).mean()), 4)
except Exception: pass
try:
    summary["deletion_auc"] = round(float(Ff.del_auc.mean()), 4)
    summary["insertion_auc"] = round(float(Ff.ins_auc.mean()), 4)
    summary["deletion_auc_random"] = round(float(Ff.del_rand.mean()), 4)
    summary["insertion_auc_random"] = round(float(Ff.ins_rand.mean()), 4)
except Exception: pass
if loaded:
    summary["final_ensemble"] = {k: round(v,4) for k,v in best.items()
                                 if not isinstance(v, list)}
(CFG.OUT_ROOT/"summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

zp = CFG.OUT_ROOT.parent/"results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for sub in ["figures","tables"]:
        for f in sorted((CFG.OUT_ROOT/sub).rglob("*")):
            if f.is_file(): z.write(f, f"results/{sub}/{f.name}")
    z.write(CFG.OUT_ROOT/"summary.json", "results/summary.json")
    for f in sorted((CFG.OUT_ROOT/"preds").glob("*.npz")):
        z.write(f, f"results/preds/{f.name}")
print(f"\nwrote {zp} ({zp.stat().st_size/1e6:.1f} MB)")


{
  "generated": "2026-09-09 06:16:28",
  "gpu": "NVIDIA GeForce RTX 3060 Ti",
  "n_images": 25331,
  "n_lesion_groups": 13311,
  "split": {
    "train": 16887,
    "val": 4222,
    "test": 4222
  },
  "ablation_at": 224,
  "final_at": 384,
  "kept_morphology": true,
  "kept_metadata": true,
  "isic2019_winner_balanced_accuracy": 0.636,
  "amil_mean": 0.4378,
  "amil_lift_over_random": 0.1402,
  "deletion_auc": 0.4697,
  "insertion_auc": 0.536,
  "deletion_auc_random": 0.5256,
  "insertion_auc_random": 0.5279,
  "final_ensemble": {
    "accuracy": 0.7949,
    "balanced_accuracy": 0.7563,
    "macro_f1": 0.6772,
    "macro_auc": 0.9539
  }
}

wrote results.zip (16.2 MB)


## 21. What to report, and where the novelty is

### The accuracy story
Report top-1 **and** balanced accuracy, always together, and anchor to the ISIC 2019
winner's 0.636 balanced accuracy. State that splits are lesion-level. Those three
choices are what make a mid-0.80s top-1 credible instead of suspicious.

### The defensible novelty, in order of strength
0. **Attention-vs-accuracy decomposition of an auxiliary branch (§16.6).** The
   strongest single claim here. Ablation says whether a branch changes accuracy;
   AMIL says whether it changes where the model looks. A branch that lifts accuracy
   without moving attention is acting as a *prior*, not as new visual evidence. That
   distinction is not made anywhere in the hybrid dermoscopy literature, and it is
   exactly what a clinician needs to know before trusting a model.
1. **Which auxiliary branch earns its place.** The literature stacks handcrafted
   descriptors onto CNNs almost without exception and almost never ablates them. You
   test morphology against patient metadata under an identical FiLM-gated interface,
   with a leave-one-out design. A clean answer either way is a real contribution,
   and the negative case is the more interesting one.
2. **FiLM gating instead of raw concatenation.** Appending 5 dims to a 3328-dim
   vector cannot express "use this feature strongly". FiLM can, so the ablation
   measures the branch's information content rather than an artefact of fusion width.
3. **The stacked-correction failure mode.** Your original combined sampler, focal,
   alpha weights and smoothing, and macro F1 fell to 0.51 with AUC at 0.906. Show
   balanced softmax recovering the gap with a single principled term. That is a
   concrete, reproducible lesson.

### On the explainability section
Report AMIL **with its random baseline** and deletion/insertion **with the random-
attribution control**. Numbers without those baselines are unfalsifiable, which is the
usual failure of XAI sections. Report the model-randomisation sanity check too: if it
fails, say so and drop the heatmaps rather than publishing decoration. AMIL is the
energy-based pointing game (Score-CAM, 2020); deletion/insertion is RISE (2018); the
sanity check is Adebayo et al. (2018). Cite all three — you are applying them, not
inventing them.

### Do not claim
* The hybrid stack as novel architecture. It is not.
* Lesion-level splitting as your finding. Cassidy et al. (*Med. Image Anal.*, 2022)
  documented ISIC duplicates and removed 14,310 of them. Cite them and move on.

### Before submitting
* Run three seeds for anything you claim, and report bootstrap intervals.
* If masks touching the border came out high in Section 5, say so: it bounds how much
  the morphology branch could ever have contributed.
* Keep the failed arms in the paper. An ablation that only reports wins is not an
  ablation.
